# 🚀 Claude Code 바이브 코딩 실습 노트북

---

> **바이브 코딩(Vibe Coding)** = AI에게 자연어로 설명해서 코드를 만드는 개발 방식

| 모듈 | 내용 | 핵심 개념 |
|------|------|-----------|
| **Module 1** | 환경 설정 & 첫 대화 | API 연동 |
| **Module 2** | 바이브 코딩 기초 | 자연어 → 코드 |
| **Module 3** | 단계별 웹앱 만들기 | 점진적 기능 추가 |
| **Module 4** | CLAUDE.md 작성 | 프로젝트 컨텍스트 설계 |
| **Module 5** | 프롬프트 전략 비교 | 좋은 vs 나쁜 프롬프트 |
| **Module 6** | Plan 모드 시뮬레이션 | 계획 → 검토 → 실행 |
| **Module 7** | 종합 실습 | 전체 워크플로우 통합 |

---
## 📦 Module 1: 환경 설정

### 1-1. 패키지 설치 + API 키 설정

> 💡 **API 키 발급**: https://console.anthropic.com/
>
> Colab 왼쪽 🔑 → `보안 비밀` 에 `ANTHROPIC_API_KEY` 등록 권장

In [ ]:
# ============================================
# 📦 1-1. 패키지 설치 + API 연결 (제일 먼저 실행!)
# ============================================

!pip install anthropic -q

import os
import re
import anthropic
from IPython.display import HTML, display, Markdown

try:
    from google.colab import userdata
    api_key = userdata.get('ANTHROPIC_API_KEY')
    print('✅ Colab 보안 비밀에서 API 키를 가져왔습니다.')
except Exception:
    api_key = input('🔑 Anthropic API 키를 입력하세요: ')

os.environ['ANTHROPIC_API_KEY'] = api_key
client = anthropic.Anthropic()
print('✅ Claude API 클라이언트 준비 완료!')

### 1-2. 헬퍼 함수 정의

> ⚠️ **반드시 이 셀을 실행**해야 이후 모든 실습이 동작합니다.

In [ ]:
# ============================================
# 🛠️ 1-2. 헬퍼 함수 (반드시 실행!)
# ============================================

def ask_claude(prompt, system='', model='claude-sonnet-4-20250514', max_tokens=4096):
    """Claude에게 질문하고 텍스트 응답을 받는 함수"""
    messages = [{'role': 'user', 'content': prompt}]
    kwargs = {'model': model, 'max_tokens': max_tokens, 'messages': messages}
    if system:
        kwargs['system'] = system
    response = client.messages.create(**kwargs)
    return response.content[0].text


def extract_html(text):
    """응답에서 HTML 코드를 추출 (코드블록 유무 모두 처리)"""
    # 방법 1: ```html ... ``` 코드 블록에서 추출
    blocks = re.findall(r'```html\s*\n(.*?)```', text, re.DOTALL)
    if blocks:
        return blocks[0].strip()
    # 방법 2: ``` ... ``` 일반 코드 블록에서 추출
    blocks = re.findall(r'```\s*\n(.*?)```', text, re.DOTALL)
    if blocks:
        return blocks[0].strip()
    # 방법 3: 코드 블록 없이 HTML이 직접 온 경우
    # <!DOCTYPE 또는 <html 로 시작하는 부분부터 </html> 까지 추출
    match = re.search(r'(<!DOCTYPE[\s\S]*?</html>)', text, re.IGNORECASE)
    if match:
        return match.group(1).strip()
    # 방법 4: <div 등 HTML 태그가 포함된 경우 전체를 HTML로 간주
    if '<div' in text or '<style' in text or '<button' in text:
        return text.strip()
    # 못 찾은 경우
    return None


def show_md(text, title='Claude 응답'):
    """마크다운 형식으로 응답 표시"""
    display(Markdown('### 💬 ' + title + '\n---\n' + text))


SYSTEM_WEB = (
    '당신은 웹 개발 전문가입니다. '
    '요청받은 내용을 하나의 완전한 HTML 코드로 작성하세요. '
    '반드시 ```html 코드 블록 안에 넣어서 출력하세요. '
    '코드 블록 하나만 출력하고, 설명은 코드 블록 밖에 짧게 쓰세요.'
)


def vibe_code(prompt, system=SYSTEM_WEB):
    """바이브 코딩 핵심: 프롬프트 → 코드 생성 → 즉시 렌더링"""
    short = prompt.replace('\n', ' ')[:80]
    print('🎯 프롬프트:', short, '...')
    print('⏳ Claude가 코드를 생성 중...')
    raw = ask_claude(prompt, system=system)
    html = extract_html(raw)
    if html:
        print('✅ 코드 생성 완료! (' + str(len(html)) + ' 글자)')
        display(HTML(html))
        return html
    else:
        print('⚠️ HTML을 찾지 못했습니다. 전체 응답:')
        show_md(raw)
        return raw


print('✅ 헬퍼 함수 정의 완료!')
print('  ask_claude(prompt)   — Claude에게 질문')
print('  extract_html(text)   — HTML 추출 (4단계 폴백)')
print('  vibe_code(prompt)    — 코드 생성 + 즉시 렌더링')
print('  show_md(text, title) — 마크다운 출력')

### 1-3. 첫 대화 테스트

In [ ]:
resp = ask_claude('안녕! 나는 바이브 코딩을 배우고 있어. 한 줄로 응원해줘!')
show_md(resp, '첫 대화 테스트')

---
## ⚡ Module 2: 바이브 코딩 기초

> 핵심: **코드를 쓰지 않고, 원하는 결과를 말로 설명**

### 2-1. 프로필 카드 UI

In [ ]:
# 🎨 실습 2-1: 프로필 카드

prompt = (
    '파란 그라디언트 배경의 프로필 카드를 HTML로 만들어줘.\n'
    '- 둥근 프로필 영역 (이모지 사용, 80px)\n'
    '- 이름: "AI 개발자"\n'
    '- 직책: "바이브 코딩 마스터"\n'
    '- 기술 태그 3개: Python, Claude, Vibe Coding (둥근 뱃지)\n'
    '- 하단에 연락하기 버튼\n'
    '- 카드 너비 350px, 가운데 정렬, 그림자\n'
    '- 순수 HTML/CSS만 사용'
)

card_html = vibe_code(prompt)

### ✏️ 직접 바꿔보기!
- 이름/배경색/태그를 변경해서 다시 실행해보세요!

### 2-2. 이모지 투표 앱

In [ ]:
# 🎮 실습 2-2: 이모지 투표 앱

prompt = (
    '이모지 투표 앱을 만들어줘.\n'
    '기능: 고양이 vs 강아지 투표, +1 버튼, '
    '막대그래프 실시간 업데이트, 비율 표시, 리셋\n'
    '디자인: 크림색 배경, 카드형태, 둥근 모서리, '
    '이모지 48px, 막대 주황/파랑, transition, 450px 가운데 정렬'
)

vote_html = vibe_code(prompt)

### 2-3. 데이터 시각화 차트

In [ ]:
# 📊 실습 2-3: 프로그래밍 언어 인기 차트

prompt = (
    '프로그래밍 언어 인기순위 가로 막대차트를 만들어줘.\n'
    '데이터: Python 28%, JavaScript 22%, TypeScript 15%, '
    'Java 12%, C# 8%, Go 6%, Rust 5%, 기타 4%\n'
    '순수 HTML/CSS/JS, 호버 강조, 각 언어 대표색상, '
    '등장 애니메이션, 다크테마, 600px 가운데 정렬'
)

chart_html = vibe_code(prompt)

---
## 🏗️ Module 3: 단계별 웹앱 만들기

> **"한 번에 전체 앱 ❌ → 기능 하나씩 추가 ✅"**

### Step 1: 기본 Todo 앱

In [ ]:
# 📝 Step 1: 기본 Todo

step1_prompt = (
    '심플한 할일 목록 앱 기본 뼈대를 만들어줘.\n'
    '기능 (이것만!): 텍스트 입력 + 추가 버튼, '
    '리스트에 추가, 각 항목에 삭제(X) 버튼\n'
    '디자인: 깔끔, 흰 배경, 가운데, 최대 500px'
)

todo_v1 = vibe_code(step1_prompt)
print('\n🔖 Step 1 완료: 기본 추가/삭제')

### Step 2: 완료 체크 기능 추가

> 💡 핵심: **이전 코드를 컨텍스트로 넘기고** 기능 추가!

In [ ]:
# 📝 Step 2: 체크박스 추가

prev_code = str(todo_v1) if todo_v1 else '<!-- 이전 코드 없음 -->'
step2_prompt = (
    '아래는 현재 할일 앱 코드야:\n\n'
    '```html\n' + prev_code + '\n```\n\n'
    '여기에 추가해줘:\n'
    '1. 각 항목 왼쪽에 체크박스\n'
    '2. 체크하면 취소선 + 회색\n'
    '3. 상단에 전체/완료 카운터\n'
    '4. 완료 항목은 아래쪽으로 이동\n'
    '기존 디자인 유지하면서 자연스럽게.'
)

todo_v2 = vibe_code(step2_prompt)
print('\n🔖 Step 2 완료: 체크박스 + 카운터')

### Step 3: 필터 + 디자인 업그레이드

In [ ]:
# 📝 Step 3: 필터 + 디자인

prev_code = str(todo_v2) if todo_v2 else '<!-- 이전 코드 없음 -->'
step3_prompt = (
    '아래는 현재 할일 앱 코드야:\n\n'
    '```html\n' + prev_code + '\n```\n\n'
    '추가/변경해줘:\n'
    '기능: 필터탭(전체/진행중/완료), Enter키 추가, 빈입력 방지\n'
    '디자인: 보라색 그라디언트 배경, 카드 그림자, '
    '탭 하이라이트, 애니메이션, 호버 효과'
)

todo_v3 = vibe_code(step3_prompt)
print('\n🔖 Step 3 완료: 필터 + 디자인')

### Step 4: 우선순위 + 다크모드 → 최종 완성!

In [ ]:
# 📝 Step 4: 최종 완성

prev_code = str(todo_v3) if todo_v3 else '<!-- 이전 코드 없음 -->'
step4_prompt = (
    '아래는 현재 할일 앱 코드야:\n\n'
    '```html\n' + prev_code + '\n```\n\n'
    '최종 추가:\n'
    '1. 우선순위(빨강 높음/노랑 보통/초록 낮음) 선택\n'
    '2. 우선순위별 정렬\n'
    '3. 더블클릭 수정\n'
    '4. 오늘 날짜 표시\n'
    '5. 전체 삭제 버튼(확인 후)\n'
    '6. 다크/라이트 모드 토글(우상단)\n'
    '완성도 높은 최종 버전으로.'
)

todo_final = vibe_code(step4_prompt)
print('\n🎉 Todo 앱 최종 완성!')

### 💡 Module 3 핵심

| 단계 | 전략 | 포인트 |
|------|------|--------|
| Step 1 | 최소 기능부터 | 범위 제한 |
| Step 2 | 이전 코드 + 추가 | **컨텍스트 제공** |
| Step 3 | 기능 + 디자인 | 구체적 지시 |
| Step 4 | 마무리 | 엣지 케이스 |

---
## 📋 Module 4: CLAUDE.md 작성 실습

> Claude Code가 **매 세션마다 자동으로 읽는** 프로젝트 설명서

### 4-1. CLAUDE.md 자동 생성

In [ ]:
# 📋 4-1: CLAUDE.md 자동 생성

project_info = (
    '다음 프로젝트를 위한 CLAUDE.md를 작성해줘:\n\n'
    '프로젝트명: 우리 동네 맛집 지도\n'
    '기술: Next.js 15, TypeScript, Tailwind CSS, Supabase\n'
    '기능: 지도 마커, 리뷰 CRUD, 카테고리 필터, 사진 업로드, 별점\n'
    '규칙: 함수형 컴포넌트, Zustand, 에러처리 필수, 한국어 주석\n\n'
    '실제 Claude Code에서 바로 쓸 수 있는 마크다운 형식으로.'
)

resp = ask_claude(project_info, system='Claude Code 전문가. 실무용 CLAUDE.md 작성.')
show_md(resp, '자동 생성된 CLAUDE.md')

### 4-2. 내 프로젝트용 CLAUDE.md

In [ ]:
# 📋 4-2: 내 프로젝트용 CLAUDE.md

print('=== CLAUDE.md 5대 핵심 ===')
print('1. 프로젝트 개요 (What)')
print('2. 핵심 명령어 (How to Run)')
print('3. 코딩 규칙 (Rules)')
print('4. 프로젝트 구조 (Where)')
print('5. 주의사항 (Watch Out)')
print()

# ✏️ 바꿔보세요!
my_name = '나의 포트폴리오 사이트'
my_tech = 'React, Tailwind CSS'
my_feat = '프로젝트 갤러리, 블로그'

p = (
    '다음 프로젝트의 간결한 CLAUDE.md를 작성해줘:\n'
    '- 프로젝트: ' + my_name + '\n'
    '- 기술: ' + my_tech + '\n'
    '- 기능: ' + my_feat + '\n\n'
    '핵심 명령어, 코딩 규칙, 프로젝트 구조, 주의사항 포함. 20줄 이내.'
)

my_md = ask_claude(p)
show_md(my_md, '내 프로젝트의 CLAUDE.md')

---
## 🎯 Module 5: 프롬프트 전략 비교

> 바이브 코딩 품질 = **프롬프트 품질**

### 5-1. 나쁜 프롬프트

In [ ]:
# ❌ 나쁜 프롬프트
print('=' * 50)
print('❌ 모호한 프롬프트')
print('=' * 50)

bad_result = vibe_code('날씨 앱 만들어줘')

### 5-1b. 좋은 프롬프트

In [ ]:
# ✅ 좋은 프롬프트
print('=' * 50)
print('✅ 구체적 프롬프트')
print('=' * 50)

good = (
    '현재 날씨 대시보드 카드를 만들어줘.\n'
    '구성: 도시 서울(24px), 온도 22도(72px 볼드), '
    '날씨 이모지, 습도/풍속/체감온도, 월~금 주간예보\n'
    '디자인: 하늘색 그라디언트, 둥근 모서리 20px, '
    '그림자, 420px 가운데 정렬\n'
    '모든 데이터 하드코딩. 순수 HTML/CSS만.'
)

good_result = vibe_code(good)

### 5-2. 프롬프트 개선 공식

| 요소 | 나쁜 예 | 좋은 예 |
|------|---------|--------|
| 범위 | "앱 만들어줘" | "로그인 페이지의 이메일 필드" |
| 기술 | (없음) | "HTML/CSS만, 라이브러리 없이" |
| 디자인 | "예쁘게" | "파란 그라디언트, 모서리 12px" |
| 데이터 | (없음) | "임시 데이터 5개 하드코딩" |

### 5-3. AI에게 프롬프트 개선 요청

In [ ]:
# 🎯 5-3: 프롬프트 개선 체인

my_vague = '계산기 만들어줘'  # ← 바꿔보세요!

improve_req = (
    '아래 바이브 코딩 프롬프트를 개선해줘:\n\n'
    '"' + my_vague + '"\n\n'
    '포함: 기능 목록, 구체적 디자인(색상/크기), '
    '기술 제약, 인터랙션 설명\n'
    '개선된 프롬프트만 출력.'
)

improved = ask_claude(improve_req)
show_md(improved, '개선된 프롬프트')

print('\n🚀 개선된 프롬프트로 코드 생성!')
result = vibe_code(improved)

---
## 🧠 Module 6: Plan 모드 시뮬레이션

> 코드 바로 작성 ❌ → **계획 → 승인 → 실행** ✅

### 6-1. 계획 수립

In [ ]:
# 🧠 6-1: Plan 모드 — 계획 수립

requirement = '회사 대시보드: 매출 차트, 사용자 통계, 최근 주문, 알림'

plan_prompt = (
    '다음 요구사항의 구현 계획을 세워줘. 코드는 아직 작성 말고.\n\n'
    '요구사항: ' + requirement + '\n\n'
    '1. 전체 구조  2. 구현 순서  3. 기술  4. 데이터 구조  5. 주의점\n'
    '계획만 작성. 코드 절대 쓰지 마.'
)

print('📋 STEP 1: 계획 수립')
plan = ask_claude(plan_prompt, system='시니어 프론트엔드 개발자.')
show_md(plan, '구현 계획서')

### 6-2. 피드백 후 실행

In [ ]:
# 🧠 6-2: Review → Execute

feedback = '매출 차트는 막대, 다크 테마, 알림 빼고 인기 상품 TOP5'
print('📝 STEP 2 피드백:', feedback)
print()

exec_prompt = (
    '계획:\n' + plan + '\n\n'
    '피드백:\n' + feedback + '\n\n'
    '피드백 반영하여 대시보드를 만들어줘.\n'
    '순수 HTML/CSS/JS, 데이터 하드코딩, 다크 테마, 반응형.'
)

print('🚀 STEP 3: 코드 생성!')
dashboard = vibe_code(exec_prompt)

### 💡 Plan 모드 흐름
```
Shift-Tab → Plan 선택 → 요청 → Claude 계획 제시
→ 사용자 검토 & 피드백 → 승인 후 실행
```

---
## 🎉 Module 7: 종합 실습 — 나만의 앱

### 7-1. 아이디어 선정

In [ ]:
# 🎉 7-1: 아이디어 선정

print('🎯 만들고 싶은 앱을 선택하세요!')
print('  1. 뮤직 플레이어 UI')
print('  2. 개인 가계부 대시보드')
print('  3. 미니 게임 (타이핑 테스트)')
print('  4. 레시피 카드 컬렉션')
print('  5. 주간 시간표')
print('  6. 운동 루틴 트래커')
print('  7. 직접 입력!')

# ✏️ 원하는 앱!
app_idea = '타이핑 속도 테스트 게임'
print('\n✅ 선택:', app_idea)

### 7-2. CLAUDE.md → Plan → Build

In [ ]:
# 🎉 Phase 1: CLAUDE.md

print('📋 Phase 1: CLAUDE.md')
p1 = (
    '프로젝트 ' + app_idea + '을 위한 '
    '간결한 CLAUDE.md를 15줄 이내로 작성해줘. '
    '기술: 순수 HTML/CSS/JS, 단일 HTML 파일.'
)
claude_md = ask_claude(p1)
show_md(claude_md, 'CLAUDE.md')

In [ ]:
# 🎉 Phase 2: 구현 계획

print('📋 Phase 2: 구현 계획')
p2 = (
    '프로젝트: ' + app_idea + '\n'
    '구현 계획을 세워줘 (코드 작성 말고):\n'
    '1. 기능 목록  2. UI 레이아웃  3. 3단계 구현 순서\n'
    '간결하게.'
)
plan7 = ask_claude(p2, system='시니어 개발자.')
show_md(plan7, '구현 계획')

In [ ]:
# 🎉 Phase 3: 코드 생성!

print('🚀 Phase 3: 코드 생성!')
p3 = (
    '프로젝트: ' + app_idea + '\n\n'
    '계획:\n' + plan7 + '\n\n'
    '이 계획대로 완성도 높은 앱을 만들어줘.\n'
    '순수 HTML/CSS/JS, 하나의 HTML, '
    '인터랙티브 디자인, 애니메이션, 점수 표시'
)
final_app = vibe_code(p3)
print('\n🎉 앱 생성 완료!')

In [ ]:
# 🎉 Phase 4: 기능 추가 (선택)

mod = '다크모드 토글과 최고 기록 저장 기능 추가해줘'  # ← 바꿔보세요!

if final_app and isinstance(final_app, str) and '<' in final_app:
    p4 = (
        '아래 코드에 수정 적용해줘:\n\n'
        '수정: ' + mod + '\n\n'
        '현재 코드:\n```html\n' + final_app + '\n```\n\n'
        '수정된 전체 코드를 출력해줘.'
    )
    modified = vibe_code(p4)
else:
    print('⚠️ Phase 3를 먼저 실행하세요.')

---
## 📝 정리: 바이브 코딩 핵심

| 원칙 | 설명 |
|------|------|
| **작게 시작** | 한 번에 전체 앱 ❌ → 기능 하나씩 ✅ |
| **구체적으로** | "예쁘게" ❌ → "파란 그라디언트, 모서리 12px" ✅ |
| **컨텍스트** | CLAUDE.md로 프로젝트 규칙 미리 정의 |
| **Plan 먼저** | 복잡한 작업은 계획 → 검토 → 실행 |
| **반복 개선** | 한 번에 완벽 ❌ → 점진적 개선 ✅ |
| **검증 필수** | AI 코드는 반드시 확인 |

### 🛠️ 실제 Claude Code
```
$ cd my-project
$ claude
> /init                     ← CLAUDE.md 자동 생성
> 로그인 페이지 만들어줘      ← 바이브 코딩!
> 소셜 로그인 버튼 추가해줘   ← 기능 추가
> 커밋하고 PR 만들어줘        ← Git 연동
```

### 📚 더 공부하기
- 공식 문서: https://code.claude.com/docs/ko/overview
- 한국어 가이드: https://github.com/revfactory/claude-code-guide

---
### 🎉 수고하셨습니다! 🚀